In [197]:
import pandas as pd
import numpy as np

tariffs = pd.read_csv('tariffs_customs.csv')



In [198]:
print(tariffs.info())
tariffs.head(20)


<class 'pandas.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   HSCode              12 non-null     float64
 1   Description         12 non-null     str    
 2   OriginCountry       12 non-null     str    
 3   DestinationCountry  12 non-null     str    
 4   StandardTariffRate  12 non-null     float64
 5   EffectiveFrom       12 non-null     str    
 6   EffectiveTo         12 non-null     str    
dtypes: float64(2), str(5)
memory usage: 1.7 KB
None


,HSCode,Description,OriginCountry,DestinationCountry,StandardTariffRate,EffectiveFrom,EffectiveTo
0,8517.12,Smartphones / Mobile Devices,China,United States,0.250,2024-01-01,2026-12-31
1,8517.12,Smartphones / Mobile Devices,Vietnam,United States,0.050,2024-01-01,2026-12-31
2,8517.18,Other Transmitters / Telecom Receivers,China,United States,0.000,2024-01-01,2026-12-31
3,8517.18,Other Transmitters / Telecom Receivers,Vietnam,United States,0.000,2024-01-01,2026-12-31
4,3004.90,Pharmaceutical Packaged Medicaments,Netherlands,United States,0.020,2024-01-01,2026-12-31
5,3004.90,Pharmaceutical Packaged Medicaments,Germany,United States,0.020,2024-01-01,2026-12-31
6,306.17,Frozen Warmwater Shrimps / Seafood,Brazil,United States,0.150,2024-01-01,2026-12-31
7,306.17,Frozen Warmwater Shrimps / Seafood,Vietnam,United States,0.080,2024-01-01,2026-12-31
8,8708.29,Automotive Parts & Accessories,Japan,United States,0.025,2024-01-01,2026-12-31
9,8708.29,Automotive Parts & Accessories,China,United States,0.250,2024-01-01,2026-12-31


In [199]:
tariffs.nunique()

HSCode                6
Description           6
OriginCountry         6
DestinationCountry    1
StandardTariffRate    8
EffectiveFrom         1
EffectiveTo           1
dtype: int64

In [200]:
def clean_tariff_hs(value):
    s = str(value).strip()
    if "." in s:
        left, right = s.split(".", 1)
        left = left.zfill(4)
        right = (right + "00")[:2]
        return (left + right).zfill(6)
    return s.zfill(6)

tariffs["HSCode6"] = tariffs["HSCode"].apply(clean_tariff_hs)

In [201]:
tariffs["EffectiveFrom"] = pd.to_datetime(tariffs["EffectiveFrom"], errors="coerce")
tariffs["EffectiveTo"] = pd.to_datetime(tariffs["EffectiveTo"], errors="coerce")

In [202]:
tariffs["InvalidRateFlag"] = (
    (tariffs["StandardTariffRate"] < 0) | 
    (tariffs["StandardTariffRate"] > 1)
)

In [203]:


def clean_hs_code(value):
    s = str(value).strip()

    if s.lower() in ["nan", "none", "<na>", ""]:
        return ""

    if "." in s:
        left, right = s.split(".", 1)
        left = left.zfill(4)
        right = (right + "00")[:2]
        return (left + right).zfill(6)

    if s.isdigit():
        return s.zfill(6)

    return s
tariffs["HSCode6"] = tariffs["HSCode"].apply(clean_hs_code).astype(str).str.strip()

In [204]:
tariffs["EffectiveFrom"] = pd.to_datetime(tariffs["EffectiveFrom"], errors="coerce")
tariffs["EffectiveTo"] = pd.to_datetime(tariffs["EffectiveTo"], errors="coerce")

In [205]:
tariffs["OriginCountry"] = tariffs["OriginCountry"].astype(str).str.strip()
tariffs["DestinationCountry"] = tariffs["DestinationCountry"].astype(str).str.strip()

In [206]:
tariff_ref = (
    tariffs.sort_values("EffectiveFrom", ascending=False)
    .drop_duplicates(
        subset=["HSCode6", "OriginCountry", "DestinationCountry"],
        keep="first"
    )
)

In [207]:
ports=pd.read_csv('ports_cleaned.csv')
legs=pd.read_csv('shipment_leg_cleaned2.csv')
shipment=pd.read_csv('shpiment_cleaned2.csv')

In [208]:
legs["HSCode6"] = legs["HSCode6"].astype(str).str.strip()
legs["ClaimedOriginCountry"] = legs["ClaimedOriginCountry"].astype(str).str.strip()

In [209]:
legs["DestCountry"] = legs["DestinationPortCountry"].astype(str).str.strip()

In [210]:
# Assuming 'legs' and 'ports' are already cleaned
ports_country = ports[["PortID", "Country"]].rename(columns={"PortID": "DestinationPortID", "Country": "DestCountry"})
legs = legs.merge(ports_country, on="DestinationPortID", how="left")
pd.DataFrame(legs)

,LegID,ShipmentID,LegSequence,CarrierID,OriginPortID,DestinationPortID,Mode,DepartureTime,ArrivalTime,LegShippingCost_USD,...,ReportedESGRating,ESGScore,ESGTier,CO2FactorCategory,CarrierModeMismatchFlag,LegExceedsCarrierSLAFlag,PreviousDestinationPortClusterID,RouteBreakClusterFlag,DestCountry_x,DestCountry_y
0,LEG-SHP-100001-1,SHP-100001,1,C10,P01,P07,Ocean,2024-05-23 00:00:00,2024-05-31 04:17:26,413.86,...,A,3,High,Low Carbon,False,False,NaN,False,Vietnam,Vietnam
1,LEG-SHP-100001-2,SHP-100001,2,C06,P07,P08,Rail,2024-05-31 09:17:26,2024-06-04 01:32:37,684.86,...,B,4,Medium,Medium Carbon,False,False,P07,False,Vietnam,Vietnam
2,LEG-SHP-100001-3,SHP-100001,3,C08,P08,P12,Road,2024-06-04 13:32:37,2024-06-07 03:15:19,539.86,...,C,5,Low,Medium Carbon,False,False,P08,False,Canada,Canada
3,LEG-SHP-100002-1,SHP-100002,1,C07,P06,P03,Air,2024-04-14 17:00:00,2024-04-16 22:50:36,755.87,...,A,3,High,High Carbon,False,True,NaN,False,United States,United States
4,LEG-SHP-100003-1,SHP-100003,1,C10,P01,P07,Ocean,2024-10-23 09:00:00,2024-11-03 02:17:27,654.03,...,A,3,High,Low Carbon,False,False,NaN,False,Vietnam,Vietnam
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27026,LEG-SHP-114999-1,SHP-114999,1,C01,P06,P03,Ocean,2024-08-20 13:00:00,2024-08-31 04:39:42,890.13,...,A,3,High,Low Carbon,False,False,NaN,False,United States,United States
27027,LEG-SHP-114999-2,SHP-114999,2,C05,P03,P12,Rail,2024-08-31 07:39:42,2024-09-03 23:23:35,855.13,...,AAA,1,High,Medium Carbon,True,False,LOS_ANGELES_LONG_BEACH,False,Canada,Canada
27028,LEG-SHP-115000-1,SHP-115000,1,C03,P01,P07,Ocean,2026-01-04 13:00:00,2026-01-14 03:49:46,1436.05,...,A,3,High,Low Carbon,False,False,NaN,False,Vietnam,Vietnam
27029,LEG-SHP-115000-2,SHP-115000,2,C06,P07,P08,Rail,2026-01-14 07:49:46,2026-01-19 10:32:03,1696.05,...,B,4,Medium,Medium Carbon,False,False,P07,False,Vietnam,Vietnam


In [211]:
# Apply the same HS cleaning function to legs
legs["HSCode6"] = legs["DeclaredHSCode"].apply(clean_tariff_hs)
tariffs.head()

,HSCode,Description,OriginCountry,DestinationCountry,StandardTariffRate,EffectiveFrom,EffectiveTo,HSCode6,InvalidRateFlag
0,8517.12,Smartphones / Mobile Devices,China,United States,0.25,2024-01-01,2026-12-31,851712,False
1,8517.12,Smartphones / Mobile Devices,Vietnam,United States,0.05,2024-01-01,2026-12-31,851712,False
2,8517.18,Other Transmitters / Telecom Receivers,China,United States,0.00,2024-01-01,2026-12-31,851718,False
3,8517.18,Other Transmitters / Telecom Receivers,Vietnam,United States,0.00,2024-01-01,2026-12-31,851718,False
4,3004.90,Pharmaceutical Packaged Medicaments,Netherlands,United States,0.02,2024-01-01,2026-12-31,300490,False


In [212]:
print("legs columns:   ", legs.columns.tolist())
print("tariffs columns:", tariffs.columns.tolist())

legs columns:    ['LegID', 'ShipmentID', 'LegSequence', 'CarrierID', 'OriginPortID', 'DestinationPortID', 'Mode', 'DepartureTime', 'ArrivalTime', 'LegShippingCost_USD', 'CustomsTariffPaid_USD', 'DeclaredHSCode', 'ClaimedOriginCountry', 'DelayReason', 'DepartureDate', 'ArrivalDate', 'DepartureYear', 'DepartureMonth', 'DepartureHour', 'DepartureDayOfWeek', 'LegDurationHours', 'LegDurationDays', 'LegIDShipmentID', 'LegIDSequence', 'LegIDMismatchFlag', 'ArrivalBeforeDepartureFlag', 'FutureDepartureFlag', 'FutureArrivalFlag', 'HSCode6', 'HSCodeChapter', 'HSCodeHeading', 'InvalidHSCodeFlag', 'NegativeLegCostFlag', 'NegativeLegCustomsFlag', 'ZeroLegCostFlag', 'HasLegCustomsFlag', 'PreviousDestinationPortID', 'RouteBreakFlag', 'PreviousArrivalTime', 'GapHours', 'NegativeGapFlag', 'LongGapFlag', 'DepartureBeforePreviousArrivalFlag', 'SameOriginDestinationFlag', 'HSCodeCount_x', 'HSCodeInconsistentFlag', 'ClaimedCountryCount', 'ClaimedCountryInconsistentFlag', 'Route', 'HasDelayReason', 'OrphanS

In [213]:
print(len(legs))

27031


In [214]:
legs = legs.merge(
    tariff_ref[[
        "HSCode6",
        "OriginCountry",
        "DestinationCountry",
        "StandardTariffRate",
        "EffectiveFrom",
        "EffectiveTo"
    ]],
    left_on=[
        "HSCode6",
        "ClaimedOriginCountry",
        "DestCountry_x"
    ],
    right_on=[
        "HSCode6",
        "OriginCountry",
        "DestinationCountry"
    ],
    how="left",
    suffixes=("", "_tariff")
)

In [215]:
legs["DepartureTime"]=pd.to_datetime(legs["DepartureTime"], errors="coerce")
print(legs["DepartureTime"].dtype)
print(legs["EffectiveFrom"].dtype)
print(legs["EffectiveTo"].dtype)

datetime64[us]
datetime64[us]
datetime64[us]


In [216]:
legs["TariffDateValid"] = (
    legs["EffectiveFrom"].notna() &
    legs["EffectiveTo"].notna() &
    (legs["DepartureTime"] >= legs["EffectiveFrom"]) &
    (legs["DepartureTime"] <= legs["EffectiveTo"])
)

In [217]:


# ----------------------------------------------------
# 1. Bring DeclaredValue_USD from shipment.csv
# ----------------------------------------------------
if "DeclaredValue_USD" not in legs.columns:
    legs = legs.merge(
        shipment[["ShipmentID", "DeclaredValue_USD"]],
        on="ShipmentID",
        how="left"
    )


# ----------------------------------------------------
# 2. Create one clean DestCountry column
# ----------------------------------------------------
if "DestCountry_y" in legs.columns:
    legs["DestCountry"] = legs["DestCountry_y"]

elif "DestCountry_x" in legs.columns:
    legs["DestCountry"] = legs["DestCountry_x"]

elif "DestinationPortCountry" in legs.columns:
    legs["DestCountry"] = legs["DestinationPortCountry"]

else:
    legs = legs.merge(
        ports[["PortID", "Country"]].rename(columns={
            "PortID": "DestinationPortID",
            "Country": "DestCountry"
        }),
        on="DestinationPortID",
        how="left"
    )

legs["DestCountry"] = legs["DestCountry"].astype(str).str.strip()


# ----------------------------------------------------
# 3. Clean merge keys
# ----------------------------------------------------
legs["HSCode6"] = legs["HSCode6"].astype(str).str.strip()
legs["ClaimedOriginCountry"] = legs["ClaimedOriginCountry"].astype(str).str.strip()

tariffs["HSCode6"] = tariffs["HSCode6"].astype(str).str.strip()
tariffs["OriginCountry"] = tariffs["OriginCountry"].astype(str).str.strip()
tariffs["DestinationCountry"] = tariffs["DestinationCountry"].astype(str).str.strip()

tariffs["EffectiveFrom"] = pd.to_datetime(tariffs["EffectiveFrom"], errors="coerce")
tariffs["EffectiveTo"] = pd.to_datetime(tariffs["EffectiveTo"], errors="coerce")


# ----------------------------------------------------
# 4. Use one tariff rate per HSCode + Origin + Destination
# ----------------------------------------------------
tariff_ref = (
    tariffs.sort_values("EffectiveFrom", ascending=False)
    .drop_duplicates(
        subset=["HSCode6", "OriginCountry", "DestinationCountry"],
        keep="first"
    )
)


# ----------------------------------------------------
# 5. Merge tariff rate if not already present
# ----------------------------------------------------
if "StandardTariffRate" not in legs.columns:
    legs = legs.merge(
        tariff_ref[[
            "HSCode6",
            "OriginCountry",
            "DestinationCountry",
            "StandardTariffRate",
            "EffectiveFrom",
            "EffectiveTo"
        ]],
        left_on=[
            "HSCode6",
            "ClaimedOriginCountry",
            "DestCountry"
        ],
        right_on=[
            "HSCode6",
            "OriginCountry",
            "DestinationCountry"
        ],
        how="left",
        suffixes=("", "_tariff")
    )


# ----------------------------------------------------
# 6. Convert date columns to datetime
# ----------------------------------------------------
legs["DepartureTime"] = (
    pd.to_datetime(legs["DepartureTime"], errors="coerce")
    .astype("datetime64[ns]")
)

legs["EffectiveFrom"] = (
    pd.to_datetime(legs["EffectiveFrom"], errors="coerce")
    .astype("datetime64[ns]")
)

legs["EffectiveTo"] = (
    pd.to_datetime(legs["EffectiveTo"], errors="coerce")
    .astype("datetime64[ns]")
)


# ----------------------------------------------------
# 7. Check tariff effective date validity
# ----------------------------------------------------
legs["TariffDateValid"] = (
    legs["EffectiveFrom"].notna() &
    legs["EffectiveTo"].notna() &
    legs["DepartureTime"].notna() &
    (legs["DepartureTime"] >= legs["EffectiveFrom"]) &
    (legs["DepartureTime"] <= legs["EffectiveTo"])
)


# ----------------------------------------------------
# 8. Calculate expected customs
# ----------------------------------------------------
legs["ExpectedCustoms_USD"] = (
    legs["DeclaredValue_USD"] * legs["StandardTariffRate"]
)

In [218]:
# --------------------------------------------------
# 1. Remove old DeclaredValue merge artifacts
# --------------------------------------------------
old_declared_columns = [
    "DeclaredValue_USD",
    "DeclaredValue_USD_x",
    "DeclaredValue_USD_y"
]

legs = legs.drop(
    columns=[col for col in old_declared_columns if col in legs.columns],
    errors="ignore"
)


# --------------------------------------------------
# 2. Merge Shipment Declared Value safely
# --------------------------------------------------
shipment_declared = shipment[["ShipmentID", "DeclaredValue_USD"]].drop_duplicates(subset="ShipmentID")

legs = legs.merge(
    shipment_declared,
    on="ShipmentID",
    how="left"
)


# --------------------------------------------------
# 3. Clean StandardTariffRate suffixes if they exist
# --------------------------------------------------
if "StandardTariffRate" not in legs.columns:
    if "StandardTariffRate_y" in legs.columns:
        legs["StandardTariffRate"] = legs["StandardTariffRate_y"]
    elif "StandardTariffRate_x" in legs.columns:
        legs["StandardTariffRate"] = legs["StandardTariffRate_x"]

legs = legs.drop(
    columns=["StandardTariffRate_x", "StandardTariffRate_y"],
    errors="ignore"
)


# --------------------------------------------------
# 4. Check if StandardTariffRate exists
# --------------------------------------------------
if "StandardTariffRate" not in legs.columns:
    raise KeyError(
        "StandardTariffRate is missing. Merge tariffs_customs.csv into legs first."
    )


# --------------------------------------------------
# 5. Calculate Expected Customs for each leg
# --------------------------------------------------
legs["ExpectedCustoms_USD"] = (
    legs["DeclaredValue_USD"] * legs["StandardTariffRate"]
)


# --------------------------------------------------
# 6. Flag if Actual Paid is wildly different from Expected
# --------------------------------------------------
legs["CustomsOverchargedFlag"] = (
    legs["ExpectedCustoms_USD"].notna() &
    legs["CustomsTariffPaid_USD"].notna() &
    (legs["CustomsTariffPaid_USD"] > (legs["ExpectedCustoms_USD"] + 1))
)

In [219]:
print(len(legs))
print(legs["LegID"].nunique())

27031
27031


In [220]:
ports=pd.read_csv('ports_cleaned.csv')
carriers=pd.read_csv('carriers_cleaned.csv')

In [221]:
print("shipment columns:   ", shipment.columns.tolist())
print("ports columns:", ports.columns.tolist())
print("legs columns:   ", legs.columns.tolist())
print("tariffs columns:", tariffs.columns.tolist())
print("carriers columns:   ", carriers.columns.tolist())



shipment columns:    ['ShipmentID', 'CargoType', 'StorageRequirement', 'TotalWeight_KG', 'DeclaredValue_USD', 'BookingDate', 'ScheduledDeliveryDate', 'ActualDeliveryDate', 'FinalStatus', 'TotalShippingCost_USD', 'TotalCustomsPaid_USD', 'SLALatePenaltyFines_USD', 'IsTransshipment', 'LegCount_x', 'TotalLegShippingCost_USD_x', 'TotalLegCustomsPaid_USD_x', 'FirstLegDepartureTime_x', 'LastLegArrivalTime_x', 'FirstOriginPortID_x', 'FinalDestinationPortID_x', 'ModeCount_x', 'HasLegDelayReason_x', 'RouteBreakCount_x', 'DepartureBeforePreviousArrivalCount_x', 'FutureArrivalLegCount_x', 'HSCodeInconsistentFlag_x', 'ClaimedCountryInconsistentFlag_x', 'IsMultimodal_x', 'NoLegsFlag', 'LegCount_y', 'TotalLegShippingCost_USD_y', 'TotalLegCustomsPaid_USD_y', 'FirstLegDepartureTime_y', 'LastLegArrivalTime_y', 'FirstOriginPortID_y', 'FinalDestinationPortID_y', 'ModeCount_y', 'HasLegDelayReason_y', 'RouteBreakCount_y', 'DepartureBeforePreviousArrivalCount_y', 'FutureArrivalLegCount_y', 'HSCodeInconsisten

In [222]:


# --------------------------------------------------
# 1. Keep only base shipment columns
# --------------------------------------------------
base_shipment_columns = [
    "ShipmentID",
    "CargoType",
    "StorageRequirement",
    "TotalWeight_KG",
    "DeclaredValue_USD",
    "BookingDate",
    "ScheduledDeliveryDate",
    "ActualDeliveryDate",
    "FinalStatus",
    "TotalShippingCost_USD",
    "TotalCustomsPaid_USD",
    "SLALatePenaltyFines_USD",
    "IsTransshipment"
]

missing_columns = [
    col for col in base_shipment_columns
    if col not in shipment.columns
]

if missing_columns:
    raise KeyError(f"Missing base shipment columns: {missing_columns}")

shipment_clean = shipment[base_shipment_columns].drop_duplicates(subset="ShipmentID").copy()

In [223]:
# --------------------------------------------------
# 2. Clean IDs and text columns
# --------------------------------------------------
shipment_clean["ShipmentID"] = shipment_clean["ShipmentID"].astype(str).str.strip()
shipment_clean["CargoType"] = shipment_clean["CargoType"].astype(str).str.strip()
shipment_clean["StorageRequirement"] = shipment_clean["StorageRequirement"].astype(str).str.strip()
shipment_clean["FinalStatus"] = shipment_clean["FinalStatus"].astype(str).str.strip()

shipment_clean["IsTransshipment"] = (
    shipment_clean["IsTransshipment"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({
        "yes": "Yes",
        "y": "Yes",
        "true": "Yes",
        "1": "Yes",
        "no": "No",
        "n": "No",
        "false": "No",
        "0": "No"
    })
    .fillna("Unknown")
)

In [224]:
# --------------------------------------------------
# 3. Convert dates
# --------------------------------------------------
date_columns = [
    "BookingDate",
    "ScheduledDeliveryDate",
    "ActualDeliveryDate"
]

for col in date_columns:
    shipment_clean[col] = (
        pd.to_datetime(shipment_clean[col], errors="coerce")
        .astype("datetime64[ns]")
    )

In [225]:
# --------------------------------------------------
# 4. Delivery performance columns
# --------------------------------------------------
shipment_clean["TransitDays"] = np.where(
    shipment_clean["BookingDate"].notna() & shipment_clean["ActualDeliveryDate"].notna(),
    (shipment_clean["ActualDeliveryDate"] - shipment_clean["BookingDate"]).dt.days,
    np.nan
)

shipment_clean["DelayDays"] = np.where(
    shipment_clean["ScheduledDeliveryDate"].notna() & shipment_clean["ActualDeliveryDate"].notna(),
    (shipment_clean["ActualDeliveryDate"] - shipment_clean["ScheduledDeliveryDate"]).dt.days,
    np.nan
)

shipment_clean["IsLate"] = shipment_clean["DelayDays"] > 0
shipment_clean["IsOnTime"] = shipment_clean["DelayDays"] <= 0
shipment_clean["HasPenalty"] = shipment_clean["SLALatePenaltyFines_USD"] > 0

In [226]:
# --------------------------------------------------
# 5. Cost columns
# --------------------------------------------------
shipment_clean["TotalCost_USD"] = (
    shipment_clean["TotalShippingCost_USD"]
    + shipment_clean["TotalCustomsPaid_USD"]
    + shipment_clean["SLALatePenaltyFines_USD"]
)

shipment_clean["ShippingCostPerKG_USD"] = np.where(
    shipment_clean["TotalWeight_KG"] > 0,
    shipment_clean["TotalShippingCost_USD"] / shipment_clean["TotalWeight_KG"],
    0
)

shipment_clean["TotalCostPerKG_USD"] = np.where(
    shipment_clean["TotalWeight_KG"] > 0,
    shipment_clean["TotalCost_USD"] / shipment_clean["TotalWeight_KG"],
    0
)

shipment_clean["CustomsPercentOfValue"] = np.where(
    shipment_clean["DeclaredValue_USD"] > 0,
    shipment_clean["TotalCustomsPaid_USD"] / shipment_clean["DeclaredValue_USD"],
    0
)

shipment_clean["HighCustomsFlag"] = shipment_clean["CustomsPercentOfValue"] > 0.30

In [227]:
# --------------------------------------------------
# 6. Shipment date flags
# --------------------------------------------------
shipment_clean["BookingDateMissing"] = shipment_clean["BookingDate"].isna()
shipment_clean["ScheduledDateMissing"] = shipment_clean["ScheduledDeliveryDate"].isna()
shipment_clean["ActualDeliveryDateMissing"] = shipment_clean["ActualDeliveryDate"].isna()

shipment_clean["DeliveredButNoActualDateFlag"] = (
    (shipment_clean["FinalStatus"].str.lower() == "delivered") &
    (shipment_clean["ActualDeliveryDate"].isna())
)

today = pd.Timestamp.today().normalize()

shipment_clean["FutureActualDeliveryFlag"] = shipment_clean["ActualDeliveryDate"] > today

In [228]:
# --------------------------------------------------
# 7. Prepare legs for aggregation
# --------------------------------------------------
legs_summary_source = legs.sort_values(["ShipmentID", "LegSequence"]).copy()

boolean_leg_columns = [
    "HasDelayReason",
    "RouteBreakFlag",
    "RouteBreakClusterFlag",
    "DepartureBeforePreviousArrivalFlag",
    "FutureArrivalFlag",
    "HSCodeInconsistentFlag",
    "ClaimedCountryInconsistentFlag",
    "CarrierModeMismatchFlag",
    "LegExceedsCarrierSLAFlag",
    "CustomsOverchargedFlag"
]

for col in boolean_leg_columns:
    if col in legs_summary_source.columns:
        legs_summary_source[col] = (
            legs_summary_source[col]
            .fillna(False)
            .astype(bool)
        )

In [229]:
# --------------------------------------------------
# 8. Aggregate legs to shipment level
# --------------------------------------------------
leg_summary = legs_summary_source.groupby("ShipmentID").agg(
    LegCount=("LegID", "count"),
    TotalLegShippingCost_USD=("LegShippingCost_USD", "sum"),
    TotalLegCustomsPaid_USD=("CustomsTariffPaid_USD", "sum"),
    FirstLegDepartureTime=("DepartureTime", "min"),
    LastLegArrivalTime=("ArrivalTime", "max"),
    FirstOriginPortID=("OriginPortID", "first"),
    FinalDestinationPortID=("DestinationPortID", "last"),
    ModeCount=("Mode", "nunique"),
    HasLegDelayReason=("HasDelayReason", "max"),
    RouteBreakPortCount=("RouteBreakFlag", "sum"),
    RouteBreakClusterCount=("RouteBreakClusterFlag", "sum"),
    DepartureBeforePreviousArrivalCount=("DepartureBeforePreviousArrivalFlag", "sum"),
    FutureArrivalLegCount=("FutureArrivalFlag", "sum"),
    HSCodeInconsistentFlag=("HSCodeInconsistentFlag", "max"),
    ClaimedCountryInconsistentFlag=("ClaimedCountryInconsistentFlag", "max"),
    CarrierModeMismatchCount=("CarrierModeMismatchFlag", "sum"),
    LegExceedsCarrierSLACount=("LegExceedsCarrierSLAFlag", "sum"),
    CustomsOverchargedLegCount=("CustomsOverchargedFlag", "sum")
).reset_index()

leg_summary["IsMultimodal"] = leg_summary["ModeCount"] > 1

In [230]:
# --------------------------------------------------
# 9. Merge leg summary once
# --------------------------------------------------
shipment_clean = shipment_clean.merge(
    leg_summary,
    on="ShipmentID",
    how="left"
)

In [231]:
shipment_clean["LegCount"] = shipment_clean["LegCount"].fillna(0).astype(int)

numeric_leg_columns = [
    "TotalLegShippingCost_USD",
    "TotalLegCustomsPaid_USD",
    "ModeCount",
    "RouteBreakPortCount",
    "RouteBreakClusterCount",
    "DepartureBeforePreviousArrivalCount",
    "FutureArrivalLegCount",
    "CarrierModeMismatchCount",
    "LegExceedsCarrierSLACount",
    "CustomsOverchargedLegCount"
]

for col in numeric_leg_columns:
    if col in shipment_clean.columns:
        shipment_clean[col] = shipment_clean[col].fillna(0)

shipment_clean["HasLegDelayReason"] = shipment_clean["HasLegDelayReason"].fillna(False).astype(bool)
shipment_clean["HSCodeInconsistentFlag"] = shipment_clean["HSCodeInconsistentFlag"].fillna(False).astype(bool)
shipment_clean["ClaimedCountryInconsistentFlag"] = shipment_clean["ClaimedCountryInconsistentFlag"].fillna(False).astype(bool)
shipment_clean["IsMultimodal"] = shipment_clean["IsMultimodal"].fillna(False).astype(bool)

In [232]:
# --------------------------------------------------
# 10. No legs flag
# --------------------------------------------------
shipment_clean["NoLegsFlag"] = shipment_clean["LegCount"] == 0

# --------------------------------------------------
# 11. Cost reconciliation flags
# --------------------------------------------------
shipment_clean["ShippingCostMismatchFlag"] = (
    abs(
        shipment_clean["TotalShippingCost_USD"] -
        shipment_clean["TotalLegShippingCost_USD"]
    ) > 1
)

shipment_clean["LegCustomsMismatchFlag"] = (
    abs(
        shipment_clean["TotalCustomsPaid_USD"] -
        shipment_clean["TotalLegCustomsPaid_USD"]
    ) > 1
)

# --------------------------------------------------
# 12. Transshipment mismatch
# --------------------------------------------------
shipment_clean["TransshipmentMismatchFlag"] = (
    ((shipment_clean["IsTransshipment"] == "Yes") & (shipment_clean["LegCount"] <= 1)) |
    ((shipment_clean["IsTransshipment"] == "No") & (shipment_clean["LegCount"] > 1))
)

# --------------------------------------------------
# 13. Late shipment but no leg delay reason
# --------------------------------------------------
shipment_clean["ShipmentLateNoLegDelayReasonFlag"] = (
    (shipment_clean["IsLate"] == True) &
    (shipment_clean["HasLegDelayReason"] == False)
)

# --------------------------------------------------
# 14. Route break shipment flag
# --------------------------------------------------
shipment_clean["RouteBreakShipmentFlag"] = shipment_clean["RouteBreakClusterCount"] > 0

In [233]:
# --------------------------------------------------
# 15. Get first tariff rate per shipment
# --------------------------------------------------
tariff_rate_legs = (
    legs[legs["StandardTariffRate"].notna()]
    .sort_values(["ShipmentID", "LegSequence"])
)

shipment_tariff_rate = (
    tariff_rate_legs.groupby("ShipmentID")
    .agg(ShipmentTariffRate=("StandardTariffRate", "first"))
    .reset_index()
)

shipment_clean = shipment_clean.merge(
    shipment_tariff_rate,
    on="ShipmentID",
    how="left"
)

shipment_clean["ExpectedShipmentCustoms_USD"] = (
    shipment_clean["DeclaredValue_USD"] *
    shipment_clean["ShipmentTariffRate"]
)

shipment_clean["CustomsCalculationMismatchFlag"] = (
    shipment_clean["ExpectedShipmentCustoms_USD"].notna() &
    (
        abs(
            shipment_clean["TotalCustomsPaid_USD"] -
            shipment_clean["ExpectedShipmentCustoms_USD"]
        ) > 1
    )
)

shipment_clean["CustomsOverchargedShipmentFlag"] = (
    shipment_clean["ExpectedShipmentCustoms_USD"].notna() &
    (
        shipment_clean["TotalCustomsPaid_USD"] >
        (shipment_clean["ExpectedShipmentCustoms_USD"] + 1)
    )
)

In [234]:
# --------------------------------------------------
# 16. Clean legs duplicate/leftover columns
# --------------------------------------------------
legs = legs.drop(
    columns=[
        "DestCountry_x",
        "DestCountry_y"
    ],
    errors="ignore"
)

# Clean HSCodeCount_x
if "HSCodeCount_x" in legs.columns and "HSCodeCount" not in legs.columns:
    legs = legs.rename(columns={"HSCodeCount_x": "HSCodeCount"})
elif "HSCodeCount_x" in legs.columns:
    legs = legs.drop(columns=["HSCodeCount_x"], errors="ignore")

In [235]:
legs = legs.rename(columns={
    "OriginCountry": "TariffOriginCountry",
    "DestinationCountry": "TariffDestinationCountry"
})

In [236]:
# --------------------------------------------------
# 17. Refresh port usage
# --------------------------------------------------
used_ports = set(legs["OriginPortID"]).union(set(legs["DestinationPortID"]))

ports["UsedInLegsFlag"] = ports["PortID"].isin(used_ports)
ports["UnusedPortFlag"] = ~ports["UsedInLegsFlag"]

In [237]:
# --------------------------------------------------
# 18. Refresh carrier usage
# --------------------------------------------------
used_carriers = set(legs["CarrierID"])

carriers["UsedInLegsFlag"] = carriers["CarrierID"].isin(used_carriers)
carriers["UnusedCarrierFlag"] = ~carriers["UsedInLegsFlag"]

In [238]:
quality_summary = {
    "Total Shipments": shipment_clean["ShipmentID"].nunique(),
    "Duplicate ShipmentIDs": shipment_clean["ShipmentID"].duplicated().sum(),
    "Shipments With No Legs": shipment_clean["NoLegsFlag"].sum(),
    "Shipping Cost Mismatch": shipment_clean["ShippingCostMismatchFlag"].sum(),
    "Leg Customs Mismatch": shipment_clean["LegCustomsMismatchFlag"].sum(),
    "Transshipment Mismatch": shipment_clean["TransshipmentMismatchFlag"].sum(),
    "Late But No Leg Delay Reason": shipment_clean["ShipmentLateNoLegDelayReasonFlag"].sum(),
    "Route Break Shipments": shipment_clean["RouteBreakShipmentFlag"].sum(),
    "Customs Calculation Mismatch": shipment_clean["CustomsCalculationMismatchFlag"].sum(),
    "Customs Overcharged Shipments": shipment_clean["CustomsOverchargedShipmentFlag"].sum(),
    "High Customs Shipments": shipment_clean["HighCustomsFlag"].sum(),
    "Delivered But No Actual Date": shipment_clean["DeliveredButNoActualDateFlag"].sum(),
    "Total Legs": legs["LegID"].nunique(),
    "Duplicate LegIDs": legs["LegID"].duplicated().sum(),
    "Carrier Mode Mismatch Legs": legs["CarrierModeMismatchFlag"].sum(),
    "Route Break Cluster Legs": legs["RouteBreakClusterFlag"].sum(),
    "Customs Overcharged Legs": legs["CustomsOverchargedFlag"].sum(),
    "Unused Ports": ports["UnusedPortFlag"].sum()
}

quality_summary = pd.Series(quality_summary)
print(quality_summary)

Total Shipments                  15000
Duplicate ShipmentIDs                0
Shipments With No Legs               0
Shipping Cost Mismatch               0
Leg Customs Mismatch                 0
Transshipment Mismatch            9675
Late But No Leg Delay Reason      9455
Route Break Shipments                0
Customs Calculation Mismatch      2763
Customs Overcharged Shipments     2763
High Customs Shipments            1106
Delivered But No Actual Date         0
Total Legs                       27031
Duplicate LegIDs                     0
Carrier Mode Mismatch Legs        9675
Route Break Cluster Legs             0
Customs Overcharged Legs             0
Unused Ports                         5
dtype: int64


In [239]:
# Customs overcharge amount
shipment_clean["CustomsOverchargeAmount_USD"] = np.where(
    shipment_clean["CustomsOverchargedShipmentFlag"] == True,
    shipment_clean["TotalCustomsPaid_USD"] - shipment_clean["ExpectedShipmentCustoms_USD"],
    0
)

# Fixed transshipment flag
shipment_clean["IsTransshipment_Fixed"] = np.where(
    shipment_clean["LegCount"] > 1,
    "Yes",
    "No"
)

# Late without reason category
shipment_clean["DelayInsightCategory"] = np.select(
    [
        (shipment_clean["IsLate"] == True) & (shipment_clean["HasLegDelayReason"] == False),
        (shipment_clean["IsLate"] == True) & (shipment_clean["HasLegDelayReason"] == True),
        (shipment_clean["IsLate"] == False)
    ],
    [
        "Late - No Reason",
        "Late - With Reason",
        "On Time"
    ],
    default="Unknown"
)

In [240]:
final_summary = {
    "Total Shipments": shipment_clean["ShipmentID"].nunique(),
    "Total Legs": legs["LegID"].nunique(),
    
    "On-Time Shipments": (shipment_clean["IsOnTime"] == True).sum(),
    "Late Shipments": (shipment_clean["IsLate"] == True).sum(),
    
    "Late With Reason": (shipment_clean["DelayInsightCategory"] == "Late - With Reason").sum(),
    "Late Without Reason": (shipment_clean["DelayInsightCategory"] == "Late - No Reason").sum(),
    
    "Transshipment Mismatch": shipment_clean["TransshipmentMismatchFlag"].sum(),
    "Carrier Mode Mismatch Legs": legs["CarrierModeMismatchFlag"].sum(),
    
    "Customs Overcharged Shipments": shipment_clean["CustomsOverchargedShipmentFlag"].sum(),
    "Total Customs Overcharge USD": shipment_clean["CustomsOverchargeAmount_USD"].sum(),
    
    "High Customs Shipments": shipment_clean["HighCustomsFlag"].sum(),
    
    "Shipping Cost Mismatch": shipment_clean["ShippingCostMismatchFlag"].sum(),
    "Leg Customs Mismatch": shipment_clean["LegCustomsMismatchFlag"].sum(),
    "Route Break Shipments": shipment_clean["RouteBreakShipmentFlag"].sum()
}

print(pd.Series(final_summary))

Total Shipments                  1.500000e+04
Total Legs                       2.703100e+04
On-Time Shipments                4.581000e+03
Late Shipments                   1.041900e+04
Late With Reason                 9.640000e+02
Late Without Reason              9.455000e+03
Transshipment Mismatch           9.675000e+03
Carrier Mode Mismatch Legs       9.675000e+03
Customs Overcharged Shipments    2.763000e+03
Total Customs Overcharge USD     1.204887e+08
High Customs Shipments           1.106000e+03
Shipping Cost Mismatch           0.000000e+00
Leg Customs Mismatch             0.000000e+00
Route Break Shipments            0.000000e+00
dtype: float64


In [241]:
ports.to_csv('ports_cleaned_final.csv', index=False, encoding="utf-8-sig")
legs.to_csv('shipment_leg_cleaned_final.csv', index=False, encoding="utf-8-sig")
carriers.to_csv('carriers_cleaned_final.csv', index=False, encoding="utf-8-sig")
shipment_clean.to_csv('shipment_cleaned_final.csv', index=False, encoding="utf-8-sig")
tariffs.to_csv('tariffs_customs_cleaned_final.csv', index=False, encoding="utf-8-sig" )